In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q dagshub mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 112.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 97.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 91.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 131.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21

In [ ]:
!pip install kaggle

In [ ]:
!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/kaggle.json
! chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle competitions download -c walmart-recruiting-store-sales-forecasting
!unzip -q walmart-recruiting-store-sales-forecasting.zip

100% 2.70M/2.70M [00:00<00:00, 215MB/s]



In [ ]:
!pip install pytorch-forecasting lightning

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.3/425.3 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 61.6 MB/s eta 0:00:00


In [ ]:
!unzip -q train.csv.zip
!unzip -q stores.csv.zip
!unzip -q test.csv.zip
!unzip -q features.csv.zip

unzip:  cannot find or open stores.csv.zip, stores.csv.zip.zip or stores.csv.zip.ZIP.


In [ ]:
import pandas as pd
train = pd.read_csv("train.csv")
stores = pd.read_csv("stores.csv")
features = pd.read_csv("features.csv")

train["Date"] = pd.to_datetime(train["Date"])
features["Date"] = pd.to_datetime(features["Date"])

df = train.merge(stores, on="Store", how="left")

df = df.merge(
    features,
    on=["Store", "Date", "IsHoliday"],
    how="left"
)

In [ ]:
import dagshub
import mlflow

dagshub.init(repo_owner='tsarc21', repo_name='Walmart-Recruiting---Store-Sales-Forecasting', mlflow=True)


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=1aa9e008-4e66-48da-a2ff-da9845864180&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=dbae4f9a2c7a4814d939aee807843b9a9cc753543ca76c24c4cc0276e3c30254




Accessing as tsarc21

Initialized MLflow to track repo "tsarc21/Walmart-Recruiting---Store-Sales-Forecasting"

Repository tsarc21/Walmart-Recruiting---Store-Sales-Forecasting initialized!

In [ ]:
import pandas as pd
import numpy as np

import torch

from pytorch_forecasting import (
    TimeSeriesDataSet,
    TemporalFusionTransformer
)

from pytorch_forecasting.metrics import RMSE

import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping

import mlflow
import mlflow.pytorch

In [ ]:


import pandas as pd
import numpy as np

import torch

import lightning.pytorch as pl

from lightning.pytorch.callbacks import (
    EarlyStopping,
    LearningRateMonitor
)

from pytorch_forecasting import (
    TimeSeriesDataSet,
    TemporalFusionTransformer
)

from pytorch_forecasting.metrics import RMSE

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error
)

import mlflow


target = "Weekly_Sales"

max_encoder_length = 26
max_prediction_length = 1



df = df.sort_values(
    ["Store", "Dept", "Date"]
).reset_index(drop=True)




df["time_idx"] = (
    df["Date"]
    .rank(method="dense")
    .astype(int)
)



df["Store"] = df["Store"].astype(str)
df["Dept"] = df["Dept"].astype(str)
df["Type"] = df["Type"].astype(str)


df["IsHoliday"] = (
    df["IsHoliday"]
    .replace({
        "False": "0",
        "True": "1",
        False: "0",
        True: "1"
    })
    .astype(str)
)

In [ ]:


# Make sure Date is datetime
df["Date"] = pd.to_datetime(df["Date"])

# Sort by time
df = df.sort_values("Date").reset_index(drop=True)

# Split points
train_end = df["Date"].quantile(0.80)
val_end = df["Date"].quantile(0.90)

# Create splits
train_df = df[df["Date"] <= train_end]

val_df = df[
    (df["Date"] > train_end) &
    (df["Date"] <= val_end)
]

test_df = df[df["Date"] > val_end]

print(f"TRAIN: {train_df.shape}")
print(f"VALIDATION: {val_df.shape}")
print(f"TEST: {test_df.shape}")

TRAIN: (338738, 17)
VALIDATION: (41369, 17)
TEST: (41463, 17)


In [ ]:
cutoff = train_df["Date"].max()

val_df = df[
    df["Date"] > cutoff - pd.Timedelta(weeks=52)
]

In [ ]:


from pytorch_forecasting import TimeSeriesDataSet

train_df = train_df.sort_values(
    ["Store", "Dept", "time_idx"]
)

val_df = val_df.sort_values(
    ["Store", "Dept", "time_idx"]
)


training = TimeSeriesDataSet(

    train_df,

    time_idx="time_idx",

    target=target,

    group_ids=[
        "Store",
        "Dept"
    ],

    max_encoder_length=52,

    max_prediction_length=1,


    static_categoricals=[
        "Store",
        "Dept",
        "Type"
    ],

    time_varying_known_categoricals=[
        "IsHoliday"
    ],


    time_varying_known_reals=[

        "time_idx",

        "Temperature",

        "Fuel_Price",

        "CPI",

        "Unemployment",

        "Size"

    ],


    time_varying_unknown_reals=[

        target

    ],


    add_relative_time_idx=True,

    add_target_scales=True,

    add_encoder_length=True,

    allow_missing_timesteps=True

)




validation = TimeSeriesDataSet.from_dataset(

    training,

    val_df,

    predict=True,

    stop_randomization=True

)




print("Training samples:", len(training))
print("Validation samples:", len(validation))

/usr/local/lib/python3.12/dist-packages/pytorch_forecasting/data/timeseries/_timeseries.py:1861: UserWarning: Min encoder length and/or min_prediction_idx and/or min prediction length and/or lags are too large for 229 series/groups which therefore are not present in the dataset index. This means no predictions can be made for those series. First 10 removed groups: [{'__group_id__Store': '1', '__group_id__Dept': '51'}, {'__group_id__Store': '1', '__group_id__Dept': '77'}, {'__group_id__Store': '1', '__group_id__Dept': '78'}, {'__group_id__Store': '10', '__group_id__Dept': '51'}, {'__group_id__Store': '10', '__group_id__Dept': '77'}, {'__group_id__Store': '10', '__group_id__Dept': '78'}, {'__group_id__Store': '11', '__group_id__Dept': '48'}, {'__group_id__Store': '11', '__group_id__Dept': '50'}, {'__group_id__Store': '11', '__group_id__Dept': '77'}, {'__group_id__Store': '11', '__group_id__Dept': '78'}]
  warnings.warn(


Training samples: 180882
Validation samples: 2949


/usr/local/lib/python3.12/dist-packages/pytorch_forecasting/data/timeseries/_timeseries.py:1861: UserWarning: Min encoder length and/or min_prediction_idx and/or min prediction length and/or lags are too large for 335 series/groups which therefore are not present in the dataset index. This means no predictions can be made for those series. First 10 removed groups: [{'__group_id__Store': '1', '__group_id__Dept': '47'}, {'__group_id__Store': '1', '__group_id__Dept': '77'}, {'__group_id__Store': '1', '__group_id__Dept': '78'}, {'__group_id__Store': '1', '__group_id__Dept': '99'}, {'__group_id__Store': '10', '__group_id__Dept': '45'}, {'__group_id__Store': '10', '__group_id__Dept': '47'}, {'__group_id__Store': '10', '__group_id__Dept': '77'}, {'__group_id__Store': '10', '__group_id__Dept': '78'}, {'__group_id__Store': '11', '__group_id__Dept': '47'}, {'__group_id__Store': '11', '__group_id__Dept': '48'}]
  warnings.warn(


In [ ]:



batch_size = 128


train_loader = training.to_dataloader(

    train=True,

    batch_size=batch_size,

    num_workers=2

)



val_loader = validation.to_dataloader(

    train=False,

    batch_size=batch_size,

    num_workers=2

)

In [ ]:





mlflow.set_experiment(
    "TFT_Training"
)



with mlflow.start_run(
    run_name="TFT_Run1_Baseline"
):



    tft = TemporalFusionTransformer.from_dataset(

        training,

        learning_rate=0.03,

        hidden_size=32,

        attention_head_size=4,

        dropout=0.1,

        hidden_continuous_size=16,

        loss=RMSE(),

        reduce_on_plateau_patience=4

    )



    print(
        f"Parameters: {tft.size()/1e3:.1f}k"
    )




    early_stop = EarlyStopping(

        monitor="val_loss",

        patience=5,

        mode="min"

    )


    lr_monitor = LearningRateMonitor()



    trainer = pl.Trainer(
        max_epochs=10,
        accelerator="gpu",
        devices=1,
        enable_progress_bar=True,
        log_every_n_steps=100,
        callbacks=[
            early_stop,
            lr_monitor
        ]
    )




    trainer.fit(

        tft,

        train_loader,

        val_loader

    )




    train_prediction = tft.predict(
        train_loader
    ).cpu().numpy()


    train_actual = torch.cat(
        [
            y[0]
            for x, y in iter(train_loader)
        ]
    ).cpu().numpy()



    train_rmse = np.sqrt(
        mean_squared_error(
            train_actual.flatten(),
            train_prediction.flatten()
        )
    )


    train_mae = mean_absolute_error(
        train_actual.flatten(),
        train_prediction.flatten()
    )


    print(
        f"TRAIN RMSE: {train_rmse:.4f}"
    )

    print(
        f"TRAIN MAE: {train_mae:.4f}"
    )



    val_prediction = tft.predict(
        val_loader
    ).cpu().numpy()



    val_actual = torch.cat(
        [
            y[0]
            for x, y in iter(val_loader)
        ]
    ).cpu().numpy()



    val_rmse = np.sqrt(
        mean_squared_error(
            val_actual.flatten(),
            val_prediction.flatten()
        )
    )


    val_mae = mean_absolute_error(
        val_actual.flatten(),
        val_prediction.flatten()
    )



    print(
        f"VALIDATION RMSE: {val_rmse:.4f}"
    )

    print(
        f"VALIDATION MAE: {val_mae:.4f}"
    )




    mlflow.log_params({

        "model": "TFT",

        "hidden_size": 32,

        "dropout": 0.1,

        "learning_rate": 0.03,

        "max_encoder_length": 52,

        "max_prediction_length": 1

    })


    mlflow.log_metrics({

        "train_rmse": train_rmse,
        "train_mae": train_mae,

        "validation_rmse": val_rmse,
        "validation_mae": val_mae

    })

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Parameters: 91.9k


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ RMSE                            │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │  2.1 K │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │    352 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │  6.4 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │ 16.5 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │ 14.3 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  8.4 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  8.4 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │  2.1 K │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     64 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  5.3 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │  2.6 K │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │  2.2 K │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │  2.2 K │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │     33 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 91.9 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 91.9 K                                                                                               
Total estimated model params size (MB): 0.368                                                                      
Modules in train mode: 448                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4
